# 01 - Simple Linear Regression End-to-End

This notebook teaches simple linear regression using one input feature: `study_hours`.

The flow is:

1. Business/statistical question
2. Exploratory analysis
3. Manual OLS formula
4. scikit-learn implementation
5. statsmodels inference
6. Metrics
7. Visual diagnostics
8. Interpretation and exercises


## 1. Problem statement

We want to predict `exam_score` from `study_hours`.

The model form is:

```text
exam_score = beta_0 + beta_1 * study_hours + error
```

| Term | Meaning |
|---|---|
| `beta_0` | Intercept |
| `beta_1` | Slope |
| `error` | Actual score minus predicted score |


## 2. Import libraries


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import statsmodels.api as sm

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

DATA_DIR = Path.cwd().parent / "data" if (Path.cwd().parent / "data").exists() else Path.cwd() / "data"


## 3. Load and preview data


In [ ]:
df = pd.read_csv(DATA_DIR / "simple_linear_student_scores.csv")
df.head()


## 4. Data audit


In [ ]:
pd.DataFrame({
    "column": df.columns,
    "dtype": [str(dtype) for dtype in df.dtypes],
    "missing_count": df.isna().sum().values,
    "unique_values": df.nunique().values,
})


In [ ]:
df[['study_hours', 'attendance_pct', 'exam_score']].describe().round(2)


## 5. Visualize the raw relationship

This is the first check for linearity. If the scatter plot looks strongly curved, simple linear regression may be too weak.


In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df["study_hours"], df["exam_score"], alpha=0.8)
plt.title("Study hours vs exam score")
plt.xlabel("Study hours")
plt.ylabel("Exam score")
plt.grid(True, alpha=0.3)
plt.show()


## 6. Prepare features and target

`X` must be a 2D table. `y` is a 1D target vector.


In [ ]:
X = df[["study_hours"]]
y = df["exam_score"]

print("X shape:", X.shape)
print("y shape:", y.shape)


## 7. Train-test split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=43,
)

print("Training rows:", len(X_train))
print("Testing rows :", len(X_test))


## 8. Fit scikit-learn LinearRegression

`scikit-learn` is ideal for ML workflow and prediction.


In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)

print(f"Intercept: {model.intercept_:.4f}")
print(f"Slope    : {model.coef_[0]:.4f}")


## 9. Plot fitted regression line


In [ ]:
x_line = np.linspace(df["study_hours"].min(), df["study_hours"].max(), 100)
x_line_df = pd.DataFrame({"study_hours": x_line})
y_line = model.predict(x_line_df)

plt.figure(figsize=(8, 5))
plt.scatter(df["study_hours"], df["exam_score"], alpha=0.7, label="Actual data")
plt.plot(x_line, y_line, linewidth=2, label="Fitted line")
plt.title("Simple linear regression fitted line")
plt.xlabel("Study hours")
plt.ylabel("Exam score")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 10. Evaluate model

| Metric | Interpretation |
|---|---|
| MAE | Average absolute error |
| RMSE | Penalizes large errors more |
| R² | Fraction of target variation explained |


In [ ]:
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

metrics = pd.DataFrame({
    "metric": ["MAE", "RMSE", "R2"],
    "value": [mae, rmse, r2],
})
metrics


## 11. Manual OLS formula

For simple regression:

```text
slope = sum((x_i - mean_x)(y_i - mean_y)) / sum((x_i - mean_x)^2)
intercept = mean_y - slope * mean_x
```


In [ ]:
x = X_train["study_hours"].to_numpy()
yt = y_train.to_numpy()

manual_slope = ((x - x.mean()) * (yt - yt.mean())).sum() / ((x - x.mean()) ** 2).sum()
manual_intercept = yt.mean() - manual_slope * x.mean()

pd.DataFrame({
    "implementation": ["manual_formula", "sklearn"],
    "intercept": [manual_intercept, model.intercept_],
    "slope": [manual_slope, model.coef_[0]],
})


## 12. Prediction table


In [ ]:
prediction_table = X_test.copy()
prediction_table["actual_exam_score"] = y_test.values
prediction_table["predicted_exam_score"] = pred
prediction_table["residual"] = prediction_table["actual_exam_score"] - prediction_table["predicted_exam_score"]
prediction_table.sort_values("study_hours").round(2).head(15)


## 13. Actual vs predicted plot


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, pred, alpha=0.8)
min_value = min(y_test.min(), pred.min())
max_value = max(y_test.max(), pred.max())
plt.plot([min_value, max_value], [min_value, max_value], linewidth=2)
plt.title("Actual vs predicted exam score")
plt.xlabel("Actual exam score")
plt.ylabel("Predicted exam score")
plt.grid(True, alpha=0.3)
plt.show()


## 14. Residual diagnostics


In [ ]:
residuals = y_test - pred

plt.figure(figsize=(7, 5))
plt.scatter(pred, residuals, alpha=0.8)
plt.axhline(0, linewidth=1)
plt.title("Residuals vs predicted values")
plt.xlabel("Predicted exam score")
plt.ylabel("Residual")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))
plt.hist(residuals, bins=10, edgecolor="black")
plt.title("Residual distribution")
plt.xlabel("Residual")
plt.ylabel("Count")
plt.grid(True, alpha=0.3)
plt.show()


## 15. statsmodels OLS inference


In [ ]:
ols = sm.OLS(y_train, sm.add_constant(X_train)).fit()

pd.DataFrame({
    "coefficient": ols.params,
    "p_value": ols.pvalues,
    "lower_95": ols.conf_int()[0],
    "upper_95": ols.conf_int()[1],
})


## 16. Cross-validation


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=43)
scores = cross_validate(
    LinearRegression(),
    X,
    y,
    cv=cv,
    scoring={"r2": "r2", "mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error"},
)

cv_results = pd.DataFrame({
    "fold": range(1, 6),
    "r2": scores["test_r2"],
    "mae": -scores["test_mae"],
    "rmse": -scores["test_rmse"],
})
cv_results.round(4)


## 17. Final interpretation

Use this language:

> One additional study hour is associated with an average increase in expected exam score.

Avoid causal wording unless the data comes from a causal experiment.


## Student exercises

1. Replace `study_hours` with `attendance_pct`.
2. Build a two-feature model with `study_hours` and `attendance_pct`.
3. Change the random seed and compare metrics.
4. Add a residual plot using `study_hours` on the x-axis.
5. Write a final conclusion with metrics and limitations.
